# Experiment 3: ten-dimensional transformed Muller--Brown landscape

This benchmark embeds the two-dimensional Muller--Brown landscape into ten
dimensions by adjoining eight transverse Gaussian coordinates and applying a
fixed invertible linear transform.  Sampling occurs in the transformed
coordinates, while the metastable basin map is defined in the latent plane by
gradient flow to the local minima.

The nonlocal shell directions are latent basin-to-basin directions mapped
through the same linear transform.  Langevin, Levy-score corrected shell
jumps, and raw compound-Poisson shell jumps are compared using basin masses,
basin-TV, transitions, direct weak-observable errors, and transverse Gaussian
compatibility checks.


## Scientific workflow and outputs

The notebook defines the transformed target, constructs and caches a
gradient-flow basin map, verifies target basin masses, runs the three retained
methods, and writes purpose-specific geometry, communication, target-
compatibility, transition, and safety diagnostics.


In [ ]:
import os, math, time, warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Callable

import numpy as np
import pandas as pd
import matplotlib
from IPython.display import display
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.spatial.distance import cdist
from scipy.special import logsumexp
from scipy.sparse.csgraph import minimum_spanning_tree

try:
    from joblib import Parallel, delayed
    HAS_JOBLIB = True
except Exception:
    HAS_JOBLIB = False

PROFILE = os.environ.get("LEVY_PROFILE", "paperlite").lower()
GLOBAL_SEED = 20260123

def ensure_dir(p):
    p = Path(p)
    p.mkdir(parents=True, exist_ok=True)
    return p

def find_project_root(start=None):
    path = Path.cwd() if start is None else Path(start).resolve()
    while path.name != "levy-score-sampling-project" and path.parent != path:
        path = path.parent
    if path.name != "levy-score-sampling-project":
        raise RuntimeError("Could not locate levy-score-sampling-project from current working directory")
    return path

PROJECT_ROOT = find_project_root()
RELEASE_ROOT = ensure_dir(PROJECT_ROOT / "manuscript_clean_active" / "numerics" / "four_experiment_release")
RELEASE_TABLE_DIR = ensure_dir(RELEASE_ROOT / "tables")
RELEASE_LOG_DIR = ensure_dir(RELEASE_ROOT / "logs")
RELEASE_FIG_ROOT = ensure_dir(PROJECT_ROOT / "manuscript_clean_active" / "figures" / "four_experiment_release")
MAIN_FIG_DIR = ensure_dir(RELEASE_FIG_ROOT / "main_candidates")
APPENDIX_FIG_DIR = ensure_dir(RELEASE_FIG_ROOT / "appendix_candidates")
DIAGNOSTIC_FIG_DIR = ensure_dir(RELEASE_FIG_ROOT / "diagnostics")
MANUSCRIPT_FIG_DIR = DIAGNOSTIC_FIG_DIR
CANDIDATE_RESULT_DIR = RELEASE_ROOT
def savefig(fig, figdir, name):
    figdir = ensure_dir(figdir)
    try:
        fig.tight_layout()
    except Exception:
        pass
    stem = str(name)
    if not stem.startswith(FIG_PREFIX):
        stem = FIG_PREFIX + stem
    pdf = figdir / f"{stem}.pdf"
    fig.savefig(pdf, bbox_inches="tight")
    print("saved", pdf)
    return pdf
def method_color(m): return METHOD_COLORS.get(str(m), "#7f7f7f")
def method_marker(m): return METHOD_MARKERS.get(str(m), "o")

def effective_basin_count(basin_mass):
    p = np.asarray(basin_mass, dtype=float)
    p = p / max(p.sum(), 1e-300)
    q = p[p > 0]
    return float(np.exp(-np.sum(q*np.log(q))))

def basin_coverage(basin_mass, threshold=0.02):
    p = np.asarray(basin_mass, dtype=float)
    p = p / max(p.sum(), 1e-300)
    return float(np.mean(p > threshold))

def seed_sequence(base_seed, n):
    return np.random.SeedSequence(base_seed).spawn(n)

def symmetric_atoms_from_edges(points, edges, include_reverse=True):
    atoms = []
    for i, j in edges:
        r = np.asarray(points[j]) - np.asarray(points[i])
        atoms.append(r)
        if include_reverse:
            atoms.append(-r)
    atoms = np.asarray(atoms, dtype=float)
    weights = np.ones(len(atoms)) / len(atoms)
    return atoms, weights

def mst_edges(points):
    D = cdist(points, points)
    T = minimum_spanning_tree(D).tocoo()
    return [(int(i), int(j)) for i, j in zip(T.row, T.col)]

def kinetic_langevin_step(x, p, grad_logp_fn, eps, dt, rng, gamma=0.65, mass=0.02):
    """BAOAB kinetic Langevin step.

    The momentum has invariant law N(0, eps*mass I), and the
    position update uses velocity p/mass.  A small mass makes this a
    genuinely momentum-persistent local baseline rather than a visually
    indistinguishable overdamped clone.
    """
    force = eps * grad_logp_fn(x)
    p = p + 0.5 * dt * force
    x = x + 0.5 * dt * (p / mass)
    a = math.exp(-gamma * dt)
    p = a * p + math.sqrt(eps * mass * (1.0 - a*a)) * rng.standard_normal(p.shape)
    x = x + 0.5 * dt * (p / mass)
    force = eps * grad_logp_fn(x)
    p = p + 0.5 * dt * force
    return x, p

def tamed_euler_step(x, drift, eps, dt, rng):
    norm = np.linalg.norm(drift, axis=1, keepdims=True)
    return x + dt * drift / (1.0 + dt * norm) + np.sqrt(2 * eps * dt) * rng.standard_normal(x.shape)

def safety_clip_box(x, box):
    y = x.copy()
    for k in range(x.shape[1]):
        y[:, k] = np.clip(y[:, k], box[k, 0], box[k, 1])
    return y, float(np.mean(np.any(y != x, axis=1)))

def classify_nearest(x, centers):
    return np.argmin(cdist(np.asarray(x), np.asarray(centers)), axis=1)

def mode_masses(x, centers):
    labs = classify_nearest(x, centers)
    return np.bincount(labs, minlength=len(centers)) / len(labs)

def summarize_metrics(records, tabdir, name):
    df = pd.DataFrame(records)
    df.to_csv(tabdir / f"{name}_metrics_timeseries.csv", index=False)
    return df

def plot_basin_metrics(metrics, methods, figdir, name, title):
    fig, ax = plt.subplots(1, 3, figsize=(14.2, 4.0))
    for m in methods:
        sub = metrics[metrics.method == m].groupby("time")[["basin_TV", "coverage", "effective_basin_count"]].mean().reset_index()
        if len(sub) == 0: 
            continue
        ax[0].semilogy(sub.time, np.maximum(sub.basin_TV, 1e-5), label=m, color=method_color(m))
        ax[1].plot(sub.time, sub.coverage, label=m, color=method_color(m))
        ax[2].plot(sub.time, sub.effective_basin_count, label=m, color=method_color(m))
    ax[0].set_title("basin-TV")
    ax[1].set_title("basin coverage fraction")
    ax[2].set_title("effective basin count")
    for a in ax:
        a.set_xlabel("time")
        a.grid(alpha=0.25)
    ax[0].legend(fontsize=8)
    fig.suptitle(title)
    savefig(fig, figdir, name)
    plt.show()
plt.rcParams.update({
    "figure.dpi": 135,
    "savefig.dpi": 300,
    "font.size": 9.5,
    "axes.titlesize": 10.5,
    "axes.labelsize": 9.5,
    "legend.fontsize": 8,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "grid.alpha": 0.22,
    "lines.linewidth": 1.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

def panel_label(ax, label, x=-0.12, y=1.06):
    ax.text(x, y, label, transform=ax.transAxes, fontsize=12, fontweight="bold",
            va="top", ha="left")

def clean_axes(ax, grid=True):
    if grid:
        ax.grid(alpha=0.22, lw=0.55)
    ax.tick_params(length=3, width=0.7)
    return ax



In [ ]:
# Phase 16A shared theory-driven utilities.
import sys

RELEASE_COMMON_ROOT = RELEASE_ROOT / "common"
if str(RELEASE_COMMON_ROOT) not in sys.path:
    sys.path.insert(0, str(RELEASE_COMMON_ROOT))

from levy.jumps import (
    AtomJump,
    EdgeShellJump,
    apply_compound_poisson_atom_jumps,
    apply_compound_poisson_shell_jumps,
)
from levy.score import levy_score_atoms as phase16_levy_score_atoms, levy_score_edge_shell
from levy.metrics import (
    cost_proxy_fields,
    compute_jump_step_transition_matrix,
    compute_recorded_phase_transition_matrix,
    mode_entropy_metrics,
    transition_rows_from_counts,
)
from levy.diagnostics import energy_before_after, jump_count_summary, merge_score_diagnostics
from levy.muller import build_basin_map_labels, dwell_time_by_majority, gradient_flow_labels_with_diagnostics, lookup_basin_map_labels, mass_from_labels, round_trip_count, weak_observable_error


def legendre_unit_interval(n):
    x, w = np.polynomial.legendre.leggauss(int(n))
    return 0.5 * (x + 1.0), 0.5 * w

def shell_uniform_quadrature(h_shell, n_rho):
    x, w = np.polynomial.legendre.leggauss(int(n_rho))
    return float(h_shell) * x, 0.5 * w


from levy.plot_style import (
    METHOD_STYLES as CANONICAL_METHOD_STYLES,
    apply_plot_style,
    method_color as canonical_method_color,
    method_marker as canonical_method_marker,
)
apply_plot_style(plt)
METHOD_COLORS = {name: style.color for name, style in CANONICAL_METHOD_STYLES.items()}
METHOD_MARKERS = {name: style.marker for name, style in CANONICAL_METHOD_STYLES.items()}


In [ ]:
OUTDIR = ensure_dir(CANDIDATE_RESULT_DIR / "muller10d")
FIGDIR = MANUSCRIPT_FIG_DIR
FIG_PREFIX = "muller10d_"
TABDIR = ensure_dir(RELEASE_TABLE_DIR / "03_muller_brown_10d")
BASIN_MAP_CACHE = OUTDIR / "muller10d_basin_map_cache.npz"

# Euclidean shell half-widths for Phase16D shell-jump sensitivity checks.
SHELL_WIDTH_CANDIDATES = [0.0, 0.08, 0.2, 0.4, 0.8]

PROFILE_ALIASES = {"smoke": "debug"}
PROFILE = PROFILE_ALIASES.get(PROFILE, PROFILE)
PROFILE_CONFIGS = {
    "debug": dict(n_particles=45, n_steps=45, dt=0.0035, n_seeds=1, record_every=9,
                  theta_n=3, rho_n=3, h_shell=0.2, basin_flow_step=8e-4,
                  basin_flow_max_iter=60, basin_grid_nx=70, basin_grid_ny=60, n_jobs=1),
    "paperlite": dict(n_particles=1800, n_steps=1500, dt=0.0028, n_seeds=4, record_every=30,
                      theta_n=5, rho_n=3, h_shell=0.08, basin_flow_step=6e-4,
                      basin_flow_max_iter=180, basin_grid_nx=320, basin_grid_ny=280,
                      n_jobs=1),
    "paperlite_extended": dict(n_particles=1800, n_steps=1500, dt=0.0028, n_seeds=4, record_every=30,
                               theta_n=5, rho_n=3, h_shell=0.08, basin_flow_step=6e-4,
                               basin_flow_max_iter=180, basin_grid_nx=320, basin_grid_ny=280,
                               n_jobs=1),
    "paper": dict(n_particles=7000, n_steps=4200, dt=0.0022, n_seeds=8, record_every=35,
                  theta_n=14, rho_n=7, h_shell=0.08, basin_flow_step=5e-4,
                  basin_flow_max_iter=260, basin_grid_nx=520, basin_grid_ny=460,
                  n_jobs=min(6, os.cpu_count() or 1)),
}
if PROFILE not in PROFILE_CONFIGS:
    raise ValueError(f"Unknown LEVY_PROFILE={PROFILE!r}; expected one of {sorted(PROFILE_CONFIGS)}")
CFG = PROFILE_CONFIGS[PROFILE]

PROFILE_METHODS = {
    "debug": ["Langevin", "LSC-shell", "CP-shell"],
    "paperlite": ["Langevin", "LSC-shell", "CP-shell"],
    "paperlite_extended": ["Langevin", "LSC-atom", "CP-atom", "LSC-shell", "CP-shell"],
    "paper": ["Langevin", "Kinetic-Langevin", "LSC-atom", "CP-atom", "LSC-shell", "CP-shell", "LSC-wrong"],
}

print("PROFILE =", PROFILE)
print("methods =", PROFILE_METHODS[PROFILE])
print("quadrature theta_n =", CFG["theta_n"], "rho_n =", CFG["rho_n"])

eps = 0.50
d = 10
sigma_aux = 0.75

# Kinetic-Langevin parameters; the invariant momentum law is N(0, eps*KL_MASS I).
KL_MASS = 0.02
KL_GAMMA = 0.65

In [ ]:
# -------------------------------
# Muller-Brown potential utilities
# -------------------------------
A_MB = np.array([-200.0, -100.0, -170.0, 15.0])
a_MB = np.array([-1.0, -1.0, -6.5, 0.7])
b_MB = np.array([0.0, 0.0, 11.0, 0.6])
c_MB = np.array([-10.0, -10.0, -6.5, 0.7])
x0_MB = np.array([1.0, 0.0, -0.5, -1.0])
y0_MB = np.array([0.0, 0.5, 1.5, 1.0])

def muller_raw(q):
    q = np.asarray(q)
    x = q[..., 0][..., None]
    y = q[..., 1][..., None]
    E = a_MB*(x-x0_MB)**2 + b_MB*(x-x0_MB)*(y-y0_MB) + c_MB*(y-y0_MB)**2
    ee = np.exp(np.clip(E, -120.0, 80.0))
    return np.sum(A_MB * ee, axis=-1)

def muller_grad_raw(q):
    q = np.asarray(q)
    x = q[..., 0][..., None]
    y = q[..., 1][..., None]
    X = x - x0_MB
    Y = y - y0_MB
    E = a_MB*X**2 + b_MB*X*Y + c_MB*Y**2
    ee = np.exp(np.clip(E, -120.0, 80.0))
    gx = np.sum(A_MB * ee * (2*a_MB*X + b_MB*Y), axis=-1)
    gy = np.sum(A_MB * ee * (b_MB*X + 2*c_MB*Y), axis=-1)
    return np.stack([gx, gy], axis=-1)

_min_starts = np.array([[-0.55, 1.45], [0.62, 0.03], [-0.05, 0.47], [-1.0, 0.5], [0.0, 1.0]])
_min_raw = []
for s in _min_starts:
    res = minimize(lambda z: float(muller_raw(np.asarray(z)[None, :])), s,
                   jac=lambda z: muller_grad_raw(np.asarray(z)[None, :])[0],
                   method="BFGS", options=dict(gtol=1e-10, maxiter=500))
    _min_raw.append(res.x)
muller_minima = []
for z in _min_raw:
    if not any(np.linalg.norm(z-w) < 1e-4 for w in muller_minima):
        muller_minima.append(z)
muller_minima = np.asarray(muller_minima)
muller_minima = muller_minima[np.argsort(muller_raw(muller_minima))]
muller_raw_min = float(np.min(muller_raw(muller_minima)))
MULLER_SCALE = 50.0

def U_muller(q):
    return (muller_raw(q) - muller_raw_min) / MULLER_SCALE

def grad_U_muller(q):
    return muller_grad_raw(q) / MULLER_SCALE

In [ ]:
# Fixed linear transform.  It is deliberately non-diagonal but well-conditioned.
rng_lin = np.random.default_rng(12345)
Q, _ = np.linalg.qr(rng_lin.standard_normal((d, d)))
scales_lin = np.linspace(0.75, 1.45, d)
B = Q @ np.diag(scales_lin)
Binv = np.linalg.inv(B)
BinvT = Binv.T

def z_from_x(x):
    return np.asarray(x) @ Binv.T

def x_from_z(z):
    return np.asarray(z) @ B.T

def U10(x):
    z = z_from_x(x)
    return U_muller(z[:, :2]) + 0.5 * np.sum(z[:, 2:]**2, axis=1) / (sigma_aux**2)

def grad_U10(x):
    z = z_from_x(x)
    gz = np.zeros_like(z)
    gz[:, :2] = grad_U_muller(z[:, :2])
    gz[:, 2:] = z[:, 2:] / (sigma_aux**2)
    return gz @ Binv

def logp(x):
    return -U10(np.asarray(x)) / eps

def grad_logp(x):
    return -grad_U10(np.asarray(x)) / eps

# Latent-mode centers and transformed sampling-space centers.
latent_centers = np.zeros((len(muller_minima), d))
latent_centers[:, :2] = muller_minima
basin_centers_x = x_from_z(latent_centers)
edges = mst_edges(muller_minima)

latent_atoms_2d, atom_weights = symmetric_atoms_from_edges(muller_minima, edges)
latent_atoms = np.zeros((len(latent_atoms_2d), d))
latent_atoms[:, :2] = latent_atoms_2d
atoms_x = x_from_z(latent_atoms)
jump_main = AtomJump(atoms=atoms_x, weights=atom_weights, lam=1.0, name="transformed-Muller-MST-atom", edge_count=len(edges))
jump_latent = AtomJump(atoms=latent_atoms, weights=atom_weights, lam=1.0, name="latent-Muller-MST-atom", edge_count=len(edges))
jump_main_shell = EdgeShellJump(centers=atoms_x, weights=atom_weights, lam=1.0, h_shell=CFG["h_shell"], name="transformed-Muller-MST-shell", edge_count=len(edges))
jump_latent_shell = EdgeShellJump(centers=latent_atoms, weights=atom_weights, lam=1.0, h_shell=CFG["h_shell"], name="latent-Muller-MST-shell", edge_count=len(edges))

# Misaligned control: rotate in latent 2D before mapping. It is retained only for the paper profile.
ang = np.deg2rad(35.0)
R2 = np.array([[np.cos(ang), -np.sin(ang)], [np.sin(ang), np.cos(ang)]])
latent_wrong = np.zeros_like(latent_atoms)
latent_wrong[:, :2] = latent_atoms_2d @ R2.T
jump_wrong = AtomJump(atoms=x_from_z(latent_wrong), weights=atom_weights, lam=1.0, name="transformed-wrong", edge_count=len(edges))

# Primary basin labels come from a precomputed latent-grid gradient-flow basin map.
# basin_map_metadata includes basin_label_converged_fraction and nearest_minimum_disagreement_rate.
gx = np.linspace(-1.8, 1.2, int(CFG["basin_grid_nx"]))
gy = np.linspace(-0.4, 2.2, int(CFG["basin_grid_ny"]))
box_q = np.array([[float(gx[0]), float(gx[-1])], [float(gy[0]), float(gy[-1])]])
X, Y = np.meshgrid(gx, gy, indexing="xy")
G2 = np.column_stack([X.ravel(), Y.ravel()])
P2 = np.exp((-U_muller(G2)/eps) - logsumexp(-U_muller(G2)/eps))
nearest_labs = classify_nearest(G2, muller_minima)

def _basin_cache_matches(data):
    if "basin_map_labels" not in data.files or "gx" not in data.files or "gy" not in data.files:
        return False
    if data["gx"].shape != gx.shape or data["gy"].shape != gy.shape:
        return False
    n_basins = int(data["n_basins"]) if "n_basins" in data.files else len(muller_minima)
    return (
        tuple(data["basin_map_labels"].shape) == Y.shape
        and np.allclose(data["gx"], gx)
        and np.allclose(data["gy"], gy)
        and n_basins == len(muller_minima)
    )

if BASIN_MAP_CACHE.exists():
    try:
        cached = np.load(BASIN_MAP_CACHE, allow_pickle=False)
        if _basin_cache_matches(cached):
            basin_map_labels = cached["basin_map_labels"].astype(int)
            target_basin_masses = cached["target_basin_masses"].astype(float)
            basin_map_metadata = {"cache_status": "loaded", "cache_path": str(BASIN_MAP_CACHE.relative_to(PROJECT_ROOT))}
        else:
            basin_map_labels = None
    except Exception:
        basin_map_labels = None
else:
    basin_map_labels = None

if basin_map_labels is None:
    basin_map_labels, basin_map_metadata = build_basin_map_labels(
        G2,
        Y.shape,
        muller_minima,
        grad_U_muller,
        step=CFG["basin_flow_step"],
        max_iter=CFG["basin_flow_max_iter"],
        tol=1e-7,
        nearest_classifier=lambda z: classify_nearest(z, muller_minima),
    )
    basin_labs = basin_map_labels.ravel()
    target_basin_masses = np.bincount(basin_labs, weights=P2, minlength=len(muller_minima))
    target_basin_masses = target_basin_masses / target_basin_masses.sum()
    basin_map_metadata.update({"cache_status": "computed", "cache_path": str(BASIN_MAP_CACHE.relative_to(PROJECT_ROOT))})
    np.savez_compressed(
        BASIN_MAP_CACHE,
        gx=gx,
        gy=gy,
        basin_map_labels=basin_map_labels,
        target_basin_masses=target_basin_masses,
        n_basins=np.array(len(muller_minima), dtype=int),
    )
else:
    basin_labs = basin_map_labels.ravel()

basin_map_metadata.update({
    "target_basin_mass_grid_resolution": f"{len(gx)}x{len(gy)}",
    "basin_grid_nx": int(len(gx)),
    "basin_grid_ny": int(len(gy)),
    "basin_grid_points": int(len(G2)),
    "basin_lookup": "nearest_latent_grid_cell",
})
target_nearest_minimum_mass = np.bincount(nearest_labs, weights=P2, minlength=len(muller_minima))
target_nearest_minimum_mass = target_nearest_minimum_mass / target_nearest_minimum_mass.sum()
target_basin_mass = target_basin_masses
target_z1 = float(np.sum(G2[:,0]*P2))
target_z2 = float(np.sum(G2[:,1]*P2))
target_xnorm = float(np.sum(np.sum(x_from_z(np.column_stack([G2, np.zeros((len(G2), d-2))]))**2, axis=1)*P2)
                     + np.sum(np.linalg.eigvalsh(B[:,2:] @ B[:,2:].T)) * eps * sigma_aux**2)

print("latent Muller minima:", muller_minima)
print("MST edges:", edges)
print("target basin mass:", target_basin_mass)
print("basin-map metadata:", basin_map_metadata)
print("nearest-minimum diagnostic mass:", target_nearest_minimum_mass)
print("linear condition number:", np.linalg.cond(B))
pd.DataFrame(basin_centers_x).to_csv(TABDIR/"muller10d_basin_centers_x.csv", index=False)
pd.DataFrame(atoms_x).to_csv(TABDIR/"muller10d_jump_atoms_x.csv", index=False)
pd.DataFrame(jump_main_shell.centers).to_csv(TABDIR/"muller10d_jump_shell_centers_x.csv", index=False)
pd.DataFrame([basin_map_metadata]).to_csv(TABDIR/"muller10d_basin_label_grid_diagnostics.csv", index=False)
pd.DataFrame({"basin": np.arange(len(target_basin_masses)), "target_basin_mass": target_basin_masses}).to_csv(TABDIR/"muller10d_target_basin_masses.csv", index=False)

In [ ]:
def classify_basins_10d(x):
    z = z_from_x(x)
    return lookup_basin_map_labels(z[:, :2], gx, gy, basin_map_labels)

def classify_nearest_minimum_10d(x):
    z = z_from_x(x)
    return classify_nearest(z[:, :2], muller_minima)

def basin_masses_10d(x):
    labs = classify_basins_10d(x)
    return np.bincount(labs, minlength=len(muller_minima)) / len(labs)

def nearest_minimum_masses_10d(x):
    labs = classify_nearest_minimum_10d(x)
    return np.bincount(labs, minlength=len(muller_minima)) / len(labs)

theta_nodes, theta_weights = legendre_unit_interval(CFG["theta_n"])
rho_nodes, rho_weights = shell_uniform_quadrature(CFG["h_shell"], CFG["rho_n"])

def score_for_jump_10d(x, jump):
    if getattr(jump, "jump_type", "atom") == "shell":
        return levy_score_edge_shell(
            x, jump, U10, eps,
            theta_nodes=theta_nodes, theta_weights=theta_weights,
            rho_nodes=rho_nodes, rho_weights=rho_weights,
            log_clip=80, score_clip=200, return_diagnostics=True,
        )
    return phase16_levy_score_atoms(
        x, jump, U10, eps,
        theta_nodes=theta_nodes, theta_weights=theta_weights,
        log_clip=80, score_clip=200, return_diagnostics=True,
    )

def apply_jump_for_law_10d(x, jump, rng, dt):
    if getattr(jump, "jump_type", "atom") == "shell":
        return apply_compound_poisson_shell_jumps(x, jump, rng, dt)
    return apply_compound_poisson_atom_jumps(x, jump, rng, dt)

def simulate(method, rng_seed, x0, seed_id=None):
    rng = np.random.default_rng(rng_seed)
    x = x0.copy()
    kin_mass = CFG.get("kinetic_mass", KL_MASS)
    p = math.sqrt(eps * kin_mass) * rng.standard_normal(x.shape) if method == "Kinetic-Langevin" else None
    jump = {
        "CP-atom": jump_main,
        "LSC-atom": jump_main,
        "CP-shell": jump_main_shell,
        "LSC-shell": jump_main_shell,
        "LSC-wrong": jump_wrong,
    }.get(method)
    times, samples = [], []
    score_diag_accum = []
    jump_diag_accum = []
    jump_step_counts = np.zeros((len(muller_minima), len(muller_minima)), dtype=float)
    for k in range(CFG["n_steps"] + 1):
        if k % CFG["record_every"] == 0:
            times.append(k * CFG["dt"])
            samples.append(x.copy())
        if k == CFG["n_steps"]:
            break
        dt = CFG["dt"]
        if method == "Langevin":
            x = x + dt * eps * grad_logp(x) + np.sqrt(2*eps*dt) * rng.standard_normal(x.shape)
        elif method == "Kinetic-Langevin":
            x, p = kinetic_langevin_step(x, p, grad_logp, eps, dt, rng, gamma=CFG.get("kinetic_gamma", KL_GAMMA), mass=CFG.get("kinetic_mass", KL_MASS))
        elif method.startswith("CP-"):
            x = x + dt * eps * grad_logp(x) + np.sqrt(2*eps*dt) * rng.standard_normal(x.shape)
            before = x.copy()
            before_labels = classify_basins_10d(before)
            x, counts = apply_jump_for_law_10d(x, jump, rng, dt)
            after_labels = classify_basins_10d(x)
            jumped = counts > 0
            if np.any(jumped):
                jump_step_counts += compute_jump_step_transition_matrix(before_labels[jumped], after_labels[jumped], len(muller_minima))
            jd = jump_count_summary(counts)
            jd.update(energy_before_after(U10, before, x))
            jump_diag_accum.append(jd)
        elif method in ["LSC-atom", "LSC-shell", "LSC-wrong"]:
            S, diag = score_for_jump_10d(x, jump)
            score_diag_accum.append(diag)
            drift = eps * grad_logp(x) + S
            x = tamed_euler_step(x, drift, eps, dt, rng)
            before = x.copy()
            before_labels = classify_basins_10d(before)
            x, counts = apply_jump_for_law_10d(x, jump, rng, dt)
            after_labels = classify_basins_10d(x)
            jumped = counts > 0
            if np.any(jumped):
                jump_step_counts += compute_jump_step_transition_matrix(before_labels[jumped], after_labels[jumped], len(muller_minima))
            jd = jump_count_summary(counts)
            jd.update(energy_before_after(U10, before, x))
            jump_diag_accum.append(jd)
        else:
            raise ValueError(method)
    score_diag = merge_score_diagnostics(score_diag_accum)
    jump_diag = merge_score_diagnostics(jump_diag_accum)
    if jump is not None:
        n_rho = CFG["rho_n"] if getattr(jump, "jump_type", "atom") == "shell" else 1
        cost = cost_proxy_fields(CFG["n_steps"], CFG["n_particles"], len(jump.atoms), CFG["theta_n"], n_rho, uses_score=method.startswith("LSC-"))
        jump_diag.update(cost)
        jump_diag["jump_type"] = getattr(jump, "jump_type", "atom")
        jump_diag["h_shell"] = float(getattr(jump, "h_shell", 0.0))
    return dict(method=method, seed=seed_id, rng_seed=rng_seed, times=np.array(times), samples=samples,
                score_diag=score_diag, jump_diag=jump_diag, jump_step_counts=jump_step_counts)

methods = PROFILE_METHODS[PROFILE]
base_rng = np.random.default_rng(GLOBAL_SEED + 800)
tasks = []
for s in range(CFG["n_seeds"]):
    z0 = latent_centers[0] + np.column_stack([
        0.035 * base_rng.standard_normal((CFG["n_particles"], 2)),
        math.sqrt(eps) * sigma_aux * base_rng.standard_normal((CFG["n_particles"], d-2))
    ])
    x0 = x_from_z(z0)
    for mi, m in enumerate(methods):
        tasks.append((m, GLOBAL_SEED + 8000*s + mi, x0.copy(), s))

if HAS_JOBLIB and CFG.get("n_jobs", 1) > 1:
    all_runs = Parallel(n_jobs=CFG["n_jobs"], backend="loky")(
        delayed(simulate)(m, seed, x0, sid) for (m, seed, x0, sid) in tasks
    )
else:
    all_runs = [simulate(m, seed, x0, sid) for (m, seed, x0, sid) in tasks]
print("completed", len(all_runs), "runs")
diag_rows = []
for r in all_runs:
    row = dict(method=r["method"], seed=r["seed"], rng_seed=r["rng_seed"])
    row.update(r["score_diag"])
    row.update(r["jump_diag"])
    diag_rows.append(row)
pd.DataFrame(diag_rows).to_csv(TABDIR/"muller10d_run_diagnostics.csv", index=False)

event_rows = []
for r in all_runs:
    event_rows.extend(transition_rows_from_counts(r["jump_step_counts"], [f"basin_{i}" for i in range(len(muller_minima))], r["method"], "jump_step_basin"))
pd.DataFrame(event_rows).to_csv(TABDIR/"muller10d_basin_jump_step_transition_matrix.csv", index=False)


In [ ]:
metric_rows = []
target_aux_norm = (d-2) * eps * sigma_aux**2
target_U = float(np.sum(U_muller(G2)*P2) + 0.5*(d-2)*eps)
for r in all_runs:
    for t, xs in zip(r["times"], r["samples"]):
        z = z_from_x(xs)
        mass = basin_masses_10d(xs)
        nearest_mass = nearest_minimum_masses_10d(xs)
        ent = mode_entropy_metrics(mass)
        basin_tv = 0.5*np.sum(np.abs(mass-target_basin_mass))
        row = dict(method=r["method"], seed=r["seed"], time=t,
            basin_TV=basin_tv,
            nearest_minimum_diagnostic_TV=0.5*np.sum(np.abs(nearest_mass-target_nearest_minimum_mass)),
            coverage=basin_coverage(mass),
            entropy=ent["entropy"],
            effective_basin_count=ent["effective_mode_count"],
            weak_z1=weak_observable_error(z[:,0], target_z1),
            weak_z2=weak_observable_error(z[:,1], target_z2),
            weak_aux_norm=abs(float(np.mean(np.sum(z[:,2:]**2, axis=1))) - target_aux_norm),
            weak_U=abs(float(np.mean(U10(xs))) - target_U),
            weak_xnorm=abs(float(np.mean(np.sum(xs**2, axis=1))) - target_xnorm),
        )
        row.update({
            "jump_type": "none", "h_shell": 0.0, "n_atoms": 0, "n_theta": 0, "n_rho": 0, "rho_n": 0,
            "mean_jump_count": 0.0, "total_jump_count": 0.0, "max_jump_count": 0.0,
            "score_clip_fraction": 0.0, "logratio_clip_fraction": 0.0,
            "jump_cost_proxy": 0, "score_cost_proxy": 0, "total_cost_proxy": 0, "cost_proxy": 0,
        })
        row.update(r["jump_diag"])
        row.update(r["score_diag"])
        metric_rows.append(row)
metrics = summarize_metrics(metric_rows, TABDIR, "muller10d")
display(metrics.groupby("method")[["basin_TV","coverage","effective_basin_count","weak_aux_norm"]].tail(1))

recorded_rows = []
dwell_rows = []
for r in all_runs:
    label_series = [classify_basins_10d(xs) for xs in r["samples"]]
    C = compute_recorded_phase_transition_matrix(label_series, len(muller_minima))
    recorded_rows.extend(transition_rows_from_counts(C, [f"basin_{i}" for i in range(len(muller_minima))], r["method"], "recorded_basin"))
    dwell = dwell_time_by_majority(label_series, r["times"], len(muller_minima))
    for i, val in enumerate(dwell):
        dwell_rows.append(dict(method=r["method"], seed=r["seed"], basin=f"basin_{i}", majority_dwell_time=float(val)))
    dwell_rows.append(dict(method=r["method"], seed=r["seed"], basin="round_trip_0_to_last_to_0", majority_dwell_time=float(round_trip_count(label_series, start_state=0, target_state=len(muller_minima)-1))))
pd.DataFrame(recorded_rows).to_csv(TABDIR/"muller10d_basin_recorded_transition_matrix.csv", index=False)
pd.DataFrame(dwell_rows).to_csv(TABDIR/"muller10d_basin_dwell_roundtrip.csv", index=False)


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14.2, 4.2))
for m in methods:
    final = [r["samples"][-1] for r in all_runs if r["method"] == m][0]
    z = z_from_x(final)
    ax[0].scatter(z[:350,0], z[:350,1], s=6, alpha=0.35, label=m, color=method_color(m))
ax[0].scatter(muller_minima[:,0], muller_minima[:,1], c="white", edgecolor="k", s=55)
ax[0].set_title("final particles in latent Muller plane")
ax[0].legend(fontsize=7)
final_mass = []
for m in methods:
    vals = [basin_masses_10d(r["samples"][-1]) for r in all_runs if r["method"]==m]
    final_mass.append(np.mean(vals, axis=0))
xloc = np.arange(len(muller_minima))
for k, m in enumerate(methods):
    width = min(0.14, 0.75/len(methods))
    off = (k - (len(methods)-1)/2)*width
    ax[1].bar(xloc+off, final_mass[k], width=width, label=m, color=method_color(m), alpha=0.82)
ax[1].plot(xloc, target_basin_mass, "k--", lw=2.0, label="target")
ax[1].set_title("final basin occupancy")
ax[1].legend(fontsize=7)
for m in methods:
    sub = metrics[metrics.method == m].groupby("time")[["basin_TV"]].mean().reset_index()
    ax[2].semilogy(sub.time, np.maximum(sub.basin_TV, 1e-5), label=m, color=method_color(m))
ax[2].set_title("basin-TV")
ax[2].legend(fontsize=7)
for a in ax:
    a.grid(alpha=0.2)
savefig(fig, FIGDIR, "fig01_muller10d_latent_particles_occupancy")
plt.show()

plot_basin_metrics(metrics, methods, FIGDIR, "fig02_muller10d_basin_communication", "Transformed 10D Muller basin communication")

## Projection-level target, density, and weak-error diagnostics

The ten-dimensional example is intentionally constructed from a two-dimensional Muller-Brown latent coordinate plus eight independent Gaussian directions and a non-diagonal linear map.  The scientifically relevant pictures should therefore be shown after projecting back to the latent Muller plane, while also checking the auxiliary coordinates and transformed jump directions.

In [ ]:

# In smoke execution we close diagnostic figures after saving to avoid embedding many
# large PNG payloads in the executed notebook.  In paper modes the figures display normally.
def savefig_quick(fig, figdir, name):
    figdir = ensure_dir(figdir)
    try:
        fig.tight_layout()
    except Exception:
        pass
    stem = str(name)
    if not stem.startswith(FIG_PREFIX):
        stem = FIG_PREFIX + stem
    pdf = figdir / f"{stem}.pdf"
    fig.savefig(pdf, bbox_inches="tight")
    print("saved", pdf)
    return pdf
def plot_metric_seed_band(df, metric, methods, ax, title, floor=1e-8):
    for m in methods:
        sub = df[df.method == m]
        g = sub.groupby("time")[metric].agg(["mean", "sem"]).reset_index()
        y = np.maximum(g["mean"].to_numpy(), floor)
        se = np.nan_to_num(g["sem"].to_numpy(), nan=0.0)
        ax.semilogy(g["time"], y, color=method_color(m), lw=1.8, label=m)
        ax.fill_between(g["time"], np.maximum(y-2*se, floor), y+2*se,
                        color=method_color(m), alpha=0.13, lw=0)
    ax.set_xlabel("time")
    ax.set_title(title)
    ax.grid(alpha=0.25)

Z2mass = P2.reshape(Y.shape)
Z2dens = Z2mass / ((gx[1]-gx[0])*(gy[1]-gy[0]))
U2grid = U_muller(G2).reshape(Y.shape)
latent_labs = basin_map_labels

pair_rows = []
for i in range(len(muller_minima)):
    for j in range(i+1, len(muller_minima)):
        pair_rows.append(dict(i=i, j=j, latent_distance=float(np.linalg.norm(muller_minima[j]-muller_minima[i])),
                              transformed_distance=float(np.linalg.norm(basin_centers_x[j]-basin_centers_x[i]))))
pd.DataFrame(pair_rows).to_csv(TABDIR/"muller10d_basin_pair_distances.csv", index=False)
pd.DataFrame(dict(singular_value=np.linalg.svd(B, compute_uv=False))).to_csv(TABDIR/"muller10d_linear_transform_singular_values.csv", index=False)

fig, ax = plt.subplots(2, 3, figsize=(15.8, 8.5))
cs0 = ax[0,0].contourf(X, Y, U2grid, levels=35)
ax[0,0].scatter(muller_minima[:,0], muller_minima[:,1], c="white", edgecolor="k", s=55)
ax[0,0].set_title("latent Muller energy")
fig.colorbar(cs0, ax=ax[0,0], fraction=0.046)

cs1 = ax[0,1].contourf(X, Y, Z2dens, levels=35)
ax[0,1].scatter(muller_minima[:,0], muller_minima[:,1], c="white", edgecolor="k", s=55)
ax[0,1].set_title("latent target density")
fig.colorbar(cs1, ax=ax[0,1], fraction=0.046)

ax[0,2].contourf(X, Y, latent_labs, levels=np.arange(len(muller_minima)+1)-0.5, alpha=0.75)
ax[0,2].scatter(muller_minima[:,0], muller_minima[:,1], c="white", edgecolor="k", s=55)
ax[0,2].set_title("gradient-flow basin partition")

ax[1,0].contour(X, Y, U2grid, levels=18, linewidths=0.55, alpha=0.45)
for i, j in edges:
    a, b = muller_minima[i], muller_minima[j]
    ax[1,0].plot([a[0], b[0]], [a[1], b[1]], color="k", lw=2.0, alpha=0.75)
for r in latent_atoms_2d:
    ax[1,0].arrow(muller_minima[0,0], muller_minima[0,1], r[0], r[1],
                  width=0.004, color=method_color("LSC-atom"), alpha=0.42, length_includes_head=True)
ax[1,0].set_title("latent MST jump directions")
ax[1,0].scatter(muller_minima[:,0], muller_minima[:,1], c="white", edgecolor="k", s=55)

ax[1,1].bar(np.arange(len(scales_lin)), np.linalg.svd(B, compute_uv=False), color="0.5")
ax[1,1].set_title("singular values of linear map")
ax[1,1].set_xlabel("index")

ax[1,2].hist(np.linalg.norm(jump_main.atoms, axis=1), bins=min(12, len(jump_main.atoms)), alpha=0.75, label="correct")
ax[1,2].hist(np.linalg.norm(jump_wrong.atoms, axis=1), bins=min(12, len(jump_wrong.atoms)), alpha=0.55, label="wrong")
ax[1,2].set_title("transformed jump length distribution")
ax[1,2].legend(fontsize=8)
for a in ax.ravel():
    a.grid(alpha=0.18)
for a in [ax[0,0], ax[0,1], ax[0,2], ax[1,0]]:
    a.set_xlim(gx[0], gx[-1]); a.set_ylim(gy[0], gy[-1])
savefig_quick(fig, FIGDIR, "fig00_muller10d_target_transform_jump_geometry")
plt.show()


In [ ]:

# ---------------------------------------------------------------------
# Projection density, trajectory, and auxiliary-coordinate diagnostics
# ---------------------------------------------------------------------
def empirical_hist2d_latent(z, xedges, yedges):
    H, _, _ = np.histogram2d(z[:,0], z[:,1], bins=[xedges, yedges], density=True)
    return H.T

fig, ax = plt.subplots(2, 3, figsize=(15.8, 8.8), sharex=True, sharey=True)
axes = ax.ravel()
axes[0].contourf(X, Y, Z2dens, levels=35)
axes[0].scatter(muller_minima[:,0], muller_minima[:,1], c="white", edgecolor="k", s=45)
axes[0].set_title("target latent density")
for k, m in enumerate(methods, start=1):
    final = np.vstack([z_from_x(r["samples"][-1]) for r in all_runs if r["method"] == m])
    H = empirical_hist2d_latent(final, gx, gy)
    axes[k].contourf((gx[:-1]+gx[1:])/2, (gy[:-1]+gy[1:])/2, H, levels=30)
    axes[k].scatter(muller_minima[:,0], muller_minima[:,1], c="white", edgecolor="k", s=35)
    axes[k].set_title(f"{m}: final latent density")
for a in axes:
    a.set_xlim(gx[0], gx[-1]); a.set_ylim(gy[0], gy[-1]); a.grid(alpha=0.18)
savefig(fig, FIGDIR, "fig03_muller10d_final_latent_density_vs_target")
plt.show()

fig, ax = plt.subplots(1, len(methods), figsize=(3.4*len(methods), 3.8), sharex=True, sharey=True)
if len(methods) == 1:
    ax = [ax]
for a, m in zip(ax, methods):
    run = [r for r in all_runs if r["method"] == m][0]
    arr = np.asarray([z_from_x(s) for s in run["samples"]])
    a.contour(X, Y, U2grid, levels=18, linewidths=0.55, alpha=0.45)
    n_traj = min(10, arr.shape[1])
    for j in range(n_traj):
        a.plot(arr[:, j, 0], arr[:, j, 1], lw=0.75, alpha=0.65, color=method_color(m))
    a.scatter(muller_minima[:,0], muller_minima[:,1], c="white", edgecolor="k", s=35)
    a.set_title(m)
    a.set_xlim(gx[0], gx[-1]); a.set_ylim(gy[0], gy[-1]); a.grid(alpha=0.18)
savefig(fig, FIGDIR, "fig04_muller10d_latent_trajectories")
plt.show()

# Extra weak errors computed in latent and transformed coordinates.
extra_rows = []
target_z2 = float(np.sum(G2[:,1]*P2))
target_xnorm = float(np.sum(np.sum(x_from_z(np.column_stack([G2, np.zeros((len(G2), d-2))]))**2, axis=1)*P2)
                     + np.sum(np.linalg.eigvalsh(B[:,2:] @ B[:,2:].T)) * eps * sigma_aux**2)
for r in all_runs:
    for t, xs in zip(r["times"], r["samples"]):
        z = z_from_x(xs)
        extra_rows.append(dict(method=r["method"], seed=r["seed"], time=t,
            weak_z2=abs(float(np.mean(z[:,1])) - target_z2),
            weak_aux_mean=abs(float(np.mean(z[:,2:]))),
            weak_xnorm=abs(float(np.mean(np.sum(xs**2, axis=1))) - target_xnorm),
            aux_norm=float(np.mean(np.sum(z[:,2:]**2, axis=1))),
        ))
extra = pd.DataFrame(extra_rows)
extra.to_csv(TABDIR/"muller10d_extra_projection_metrics.csv", index=False)

fig, ax = plt.subplots(1, 4, figsize=(18.0, 4.2))
plot_metric_seed_band(metrics, "weak_aux_norm", methods, ax[0], "absolute error: auxiliary squared-norm mean")
plot_metric_seed_band(extra, "weak_aux_mean", methods, ax[1], "absolute auxiliary-coordinate mean")
plot_metric_seed_band(metrics, "weak_z1", methods, ax[2], "absolute weak error: latent q1")
plot_metric_seed_band(metrics, "weak_U", methods, ax[3], "absolute weak error: total energy")
ax[0].legend(fontsize=8)
savefig(fig, FIGDIR, "fig05_muller10d_weak_observable_errors")
plt.show()

# Final auxiliary norm distributions.
rng_ref = np.random.default_rng(909)
ref_aux_norm = eps * sigma_aux**2 * rng_ref.chisquare(df=d-2, size=6000)
fig, ax = plt.subplots(1, 1, figsize=(8.6, 4.2))
ax.hist(ref_aux_norm, bins=35, density=True, histtype="step", color="k", lw=2.0, label="target auxiliary norm")
for m in methods:
    final = np.vstack([z_from_x(r["samples"][-1]) for r in all_runs if r["method"] == m])
    vals = np.sum(final[:,2:]**2, axis=1)
    ax.hist(vals, bins=35, density=True, histtype="step", lw=1.6, color=method_color(m), label=m)
ax.set_title("auxiliary-coordinate invariant shape")
ax.set_xlabel("sum of squared auxiliary latent coordinates")
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.25)
savefig(fig, FIGDIR, "fig06_muller10d_auxiliary_norm_distribution")
plt.show()



## Expanded diagnostics for the 10D transformed Muller benchmark

This experiment should not be read as a black-box ten-dimensional scatter plot.  The target factorizes in latent coordinates into a two-dimensional Muller metastable component and eight independent Gaussian directions.  The plots below therefore check both parts separately: projected metastable transport in the latent Muller plane and invariant-shape preservation in the auxiliary Gaussian coordinates.


## Purpose-specific target compatibility summary

The next figure contains direct terminal target-compatibility quantities only:
gradient-flow basin masses, latent weak-observable error, and total-energy
weak error.  Geometry, transitions, and transverse Gaussian shape checks are
reported in separate figures.


In [ ]:
# Direct terminal target-compatibility quantities.
summary_methods = [m for m in ["Langevin", "LSC-shell", "CP-shell"] if m in methods]
term = metrics[np.isclose(metrics.time, metrics.time.max())]

fig, ax = plt.subplots(1, 3, figsize=(15.6, 4.5), constrained_layout=True)
xloc = np.arange(len(muller_minima))
width = min(0.18, 0.72/max(1, len(summary_methods)))
for j, m in enumerate(summary_methods):
    vals = np.asarray([basin_masses_10d(r["samples"][-1]) for r in all_runs if r["method"] == m])
    means = vals.mean(axis=0)
    ses = vals.std(axis=0, ddof=1)/math.sqrt(vals.shape[0]) if vals.shape[0] > 1 else np.zeros(vals.shape[1])
    off = (j - (len(summary_methods)-1)/2)*width
    ax[0].bar(xloc+off, means, width=width, yerr=ses, capsize=2,
              color=method_color(m), alpha=0.84, label=m)
ax[0].plot(xloc, target_basin_mass, "k--", lw=2.0, label="target")
ax[0].set_xticks(xloc)
ax[0].set_xlabel("gradient-flow basin index")
ax[0].set_ylabel("probability mass")
ax[0].set_title("final basin masses")
ax[0].legend(fontsize=7)

for metric, title, axi in [
    ("weak_z1", "absolute weak error: latent q1", ax[1]),
    ("weak_U", "absolute weak error: total energy", ax[2]),
]:
    means = [term[term.method == m][metric].mean() for m in summary_methods]
    ses = [term[term.method == m][metric].sem() for m in summary_methods]
    axi.bar(summary_methods, means, yerr=np.nan_to_num(ses), capsize=3,
            color=[method_color(m) for m in summary_methods], alpha=0.84)
    axi.set_yscale("log")
    axi.set_title(title)
    axi.tick_params(axis="x", rotation=30)
for j, axi in enumerate(ax):
    clean_axes(axi, grid=True)
    panel_label(axi, "abc"[j])
savefig(fig, FIGDIR, "fig03_muller10d_target_compatibility")
plt.show()


In [ ]:

# ---------------------------------------------------------------------
# Expanded 10D diagnostics: latent target, transitions, aux Gaussian checks
# ---------------------------------------------------------------------
def plot_seed_band(df, metric, methods, ax, title, floor=1e-9, semilog=True):
    for m in methods:
        sub = df[df.method == m]
        g = sub.groupby("time")[metric].agg(["mean", "sem"]).reset_index()
        y = np.asarray(g["mean"], dtype=float)
        se = np.nan_to_num(np.asarray(g["sem"], dtype=float), nan=0.0)
        if semilog:
            yy = np.maximum(y, floor)
            ax.semilogy(g.time, yy, color=method_color(m), lw=1.9, label=m)
            ax.fill_between(g.time, np.maximum(yy-2*se, floor), yy+2*se, color=method_color(m), alpha=0.13, lw=0)
        else:
            ax.plot(g.time, y, color=method_color(m), lw=1.9, label=m)
            ax.fill_between(g.time, y-2*se, y+2*se, color=method_color(m), alpha=0.13, lw=0)
    ax.set_title(title); ax.set_xlabel("time"); ax.grid(alpha=0.25)

# Latent grid for the 2D Muller factor.
gq1 = np.linspace(box_q[0,0], box_q[0,1], 220 if PROFILE != "smoke" else 70)
gq2 = np.linspace(box_q[1,0], box_q[1,1], 200 if PROFILE != "smoke" else 65)
Q1, Q2 = np.meshgrid(gq1, gq2, indexing="xy")
QG = np.column_stack([Q1.ravel(), Q2.ravel()])
LQ = -U_muller(QG)/eps
PQ = np.exp(LQ - logsumexp(LQ))
ZQ = PQ.reshape(Q1.shape)
UQ = U_muller(QG).reshape(Q1.shape)
partQ = lookup_basin_map_labels(QG, gx, gy, basin_map_labels).reshape(Q1.shape)

fig, ax = plt.subplots(2, 3, figsize=(16.0, 8.6))
ax[0,0].contourf(Q1, Q2, UQ, levels=35)
ax[0,0].scatter(muller_minima[:,0], muller_minima[:,1], c="white", edgecolor="k", s=60)
ax[0,0].set_title("latent Muller energy")
ax[0,1].contourf(Q1, Q2, ZQ, levels=35)
ax[0,1].scatter(muller_minima[:,0], muller_minima[:,1], c="white", edgecolor="k", s=60)
ax[0,1].set_title("latent target density")
ax[0,2].contourf(Q1, Q2, partQ, levels=np.arange(len(muller_minima)+1)-0.5, alpha=0.55)
ax[0,2].contour(Q1, Q2, UQ, levels=12, colors="k", linewidths=0.35, alpha=0.35)
ax[0,2].scatter(muller_minima[:,0], muller_minima[:,1], c="white", edgecolor="k", s=60)
ax[0,2].set_title("gradient-flow basin partition")

# Transform diagnostics.
ax[1,0].plot(np.linalg.svd(B, compute_uv=False), marker="o")
ax[1,0].set_title("singular values of linear map B")
ax[1,0].set_xlabel("index"); ax[1,0].set_ylabel("singular value")
ax[1,0].grid(alpha=0.25)
ax[1,1].hist(np.linalg.norm(jump_main.atoms, axis=1), bins=12, alpha=0.7, color=method_color("LSC-atom"), label="10D atom norm")
ax[1,1].hist(np.linalg.norm(jump_latent.atoms[:, :2], axis=1), bins=12, histtype="step", lw=2, color="k", label="latent atom norm")
ax[1,1].set_title("jump length before/after transform"); ax[1,1].legend(fontsize=8)
ax[1,2].scatter(muller_minima[:,0], muller_minima[:,1], c="white", edgecolor="k", s=80, zorder=3)
for i, j in edges:
    ax[1,2].plot([muller_minima[i,0], muller_minima[j,0]], [muller_minima[i,1], muller_minima[j,1]], color=method_color("LSC-atom"), lw=2.5)
for k, c in enumerate(muller_minima):
    ax[1,2].text(c[0], c[1], str(k), ha="center", va="center", fontsize=8)
ax[1,2].set_title("latent MST graph used for jump design")
for a in ax.ravel():
    if a not in [ax[1,0], ax[1,1]]:
        a.set_xlim(box_q[0]); a.set_ylim(box_q[1]); a.grid(alpha=0.14)
savefig_quick(fig, FIGDIR, "fig00_muller10d_latent_target_transform")
plt.show()

# Mode transition matrices in latent projection.
def transition_matrix_for_method_10d(method):
    C = np.zeros((len(muller_minima), len(muller_minima)), dtype=float)
    for r in all_runs:
        if r["method"] != method:
            continue
        label_series = [classify_basins_10d(xs) for xs in r["samples"]]
        for a, b in zip(label_series[:-1], label_series[1:]):
            for i in range(len(muller_minima)):
                mask = (a == i)
                if np.any(mask):
                    C[i] += np.bincount(b[mask], minlength=len(muller_minima))
    row = C.sum(axis=1, keepdims=True)
    return np.divide(C, row, out=np.zeros_like(C), where=row>0)

fig, ax = plt.subplots(2, 3, figsize=(13.8, 8.0))
for a, m in zip(ax.ravel(), methods):
    Tmat = transition_matrix_for_method_10d(m)
    im = a.imshow(Tmat, vmin=0, vmax=1, cmap="viridis")
    a.set_title(f"latent basin transitions: {m}")
    a.set_xlabel("next basin"); a.set_ylabel("current basin")
    for i in range(Tmat.shape[0]):
        for j in range(Tmat.shape[1]):
            a.text(j, i, f"{Tmat[i,j]:.2f}", ha="center", va="center", fontsize=7, color="white" if Tmat[i,j] > 0.45 else "black")
for extra in ax.ravel()[len(methods):]:
    extra.axis("off")
fig.colorbar(im, ax=ax.ravel().tolist(), fraction=0.025, pad=0.01)
savefig_quick(fig, FIGDIR, "fig01b_muller10d_latent_transition_matrices")
plt.show()

# First time every latent basin has nonnegligible mass.
cover_rows = []
thr = 0.04
for r in all_runs:
    first = np.nan
    for t, xs in zip(r["times"], r["samples"]):
        mass = basin_masses_10d(xs)
        if np.all(mass > thr):
            first = float(t); break
    cover_rows.append(dict(method=r["method"], seed=r["seed"], threshold=thr, first_all_basin_time=first))
coverage_time_df = pd.DataFrame(cover_rows)
coverage_time_df.to_csv(TABDIR/"muller10d_first_all_basin_coverage_time.csv", index=False)
display(coverage_time_df.groupby("method")["first_all_basin_time"].agg(["mean","std","count"]))


In [ ]:

# ---------------------------------------------------------------------
# Projection density and auxiliary-coordinate invariant shape
# ---------------------------------------------------------------------
def hist2d_mass(samples_q, nx=70, ny=65):
    H, xe, ye = np.histogram2d(samples_q[:,0], samples_q[:,1], bins=[nx, ny], range=[box_q[0], box_q[1]], density=False)
    H = H.T.astype(float); H /= max(H.sum(), 1.0)
    return H, xe, ye

def target_q_mass_on_bins(xe, ye):
    ix = np.clip(np.searchsorted(xe, QG[:,0], side="right") - 1, 0, len(xe)-2)
    iy = np.clip(np.searchsorted(ye, QG[:,1], side="right") - 1, 0, len(ye)-2)
    T = np.zeros((len(ye)-1, len(xe)-1), dtype=float)
    np.add.at(T, (iy, ix), PQ)
    T /= max(T.sum(), 1e-300)
    return T

fig, ax = plt.subplots(2, 3, figsize=(15.5, 8.8))
density_rows = []
for a, m in zip(ax.ravel(), methods):
    final = np.vstack([r["samples"][-1] for r in all_runs if r["method"] == m])
    q = z_from_x(final)[:, :2]
    H, xe, ye = hist2d_mass(q)
    T = target_q_mass_on_bins(xe, ye)
    l1 = float(np.sum(np.abs(H-T)))
    density_rows.append(dict(method=m, latent_grid_L1=l1))
    a.imshow(H, origin="lower", extent=[box_q[0,0], box_q[0,1], box_q[1,0], box_q[1,1]], aspect="auto", cmap="magma")
    a.contour(Q1, Q2, ZQ, levels=8, colors="cyan", linewidths=0.65, alpha=0.7)
    a.scatter(muller_minima[:,0], muller_minima[:,1], c="white", edgecolor="k", s=45)
    a.set_title(f"{m}: latent density, L1={l1:.2f}")
    a.set_xlim(box_q[0]); a.set_ylim(box_q[1]); a.grid(alpha=0.12)
for extra in ax.ravel()[len(methods):]:
    extra.axis("off")
savefig_quick(fig, FIGDIR, "fig02b_muller10d_latent_density_by_method")
plt.show()
pd.DataFrame(density_rows).to_csv(TABDIR/"muller10d_latent_density_grid_errors.csv", index=False)
display(pd.DataFrame(density_rows))

# Auxiliary coordinate distribution checks.
aux_rows = []
fig, ax = plt.subplots(1, 3, figsize=(15.8, 4.4))
# The target auxiliary squared norm has expectation target_aux_norm.
for m in methods:
    final = np.vstack([r["samples"][-1] for r in all_runs if r["method"] == m])
    z = z_from_x(final)
    aux = z[:, 2:]
    aux_norm = np.sum(aux**2, axis=1)
    aux_rows.append(dict(method=m, aux_norm_mean=float(aux_norm.mean()), aux_norm_target=float(target_aux_norm), aux_norm_abs_error=float(abs(aux_norm.mean()-target_aux_norm)), aux_coord_mean_abs=float(np.mean(np.abs(aux.mean(axis=0))))))
    ax[0].hist(aux_norm, bins=35, density=True, histtype="step", lw=1.6, color=method_color(m), label=m)
    ax[1].hist(aux[:,0], bins=35, density=True, histtype="step", lw=1.6, color=method_color(m), label=m)
    ax[2].plot(np.sort(aux_norm)[:min(1000, len(aux_norm))], lw=1.2, color=method_color(m), alpha=0.8, label=m)
ax[0].axvline(target_aux_norm, color="k", ls="--", lw=1.8, label="target mean")
ax[0].set_title("auxiliary squared norm distribution")
ax[1].set_title("first auxiliary coordinate")
ax[2].set_title("sorted auxiliary norm sample")
for a in ax:
    a.grid(alpha=0.22); a.legend(fontsize=7)
pd.DataFrame(aux_rows).to_csv(TABDIR/"muller10d_auxiliary_distribution_checks.csv", index=False)
savefig_quick(fig, FIGDIR, "fig04b_muller10d_auxiliary_distribution_checks")
plt.show()
display(pd.DataFrame(aux_rows))


## Latent basin occupancy and jump-direction diagnostics

In [ ]:

# Final latent-basin occupancy: the 10D transformed problem should inherit the 2D Muller basin structure.
mass_rows = []
for m in methods:
    vals = np.asarray([basin_masses_10d(r["samples"][-1]) for r in all_runs if r["method"] == m])
    for k in range(vals.shape[1]):
        mass_rows.append(dict(method=m, basin=k, mean=float(vals[:,k].mean()),
                              se=float(vals[:,k].std(ddof=1)/math.sqrt(max(1, vals.shape[0]))) if vals.shape[0] > 1 else 0.0,
                              target=float(target_basin_mass[k])))
mass10_df = pd.DataFrame(mass_rows)
mass10_df.to_csv(TABDIR/"muller10d_final_latent_basin_occupancy_by_method.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(13.8, 4.8), constrained_layout=True)
xloc = np.arange(len(muller_minima)); width = min(0.15, 0.75/len(methods))
for j, m in enumerate(methods):
    sub = mass10_df[mass10_df.method == m].sort_values("basin")
    off = (j - (len(methods)-1)/2)*width
    ax[0].bar(xloc+off, sub["mean"], width=width, yerr=sub["se"], capsize=2,
              color=method_color(m), alpha=0.84, label=m)
ax[0].plot(xloc, target_basin_mass, "k--", lw=2.0, label="target")
ax[0].set_xlabel("gradient-flow basin index")
ax[0].set_ylabel("probability mass")
ax[0].set_title("10D final latent-basin occupancy")
ax[0].legend(fontsize=7, ncol=2)
# Latent jump atom directions inherited from the 2D Muller problem and transformed to x-space.
latent_angles = np.arctan2(jump_latent.atoms[:,1], jump_latent.atoms[:,0])
latent_lengths = np.linalg.norm(jump_latent.atoms[:, :2], axis=1)
x_lengths = np.linalg.norm(jump_main.atoms, axis=1)
ax[1].scatter(latent_angles, x_lengths, s=74, c=latent_lengths, cmap="viridis", edgecolor="black", linewidth=0.35)
ax[1].set_xlabel("latent atom angle")
ax[1].set_ylabel("transformed 10D atom length")
ax[1].set_title("jump directions after the 10D linear transform")
for a in ax: clean_axes(a, grid=True)
savefig(fig, FIGDIR, "fig04c_muller10d_basin_occupancy_and_jump_audit")
plt.show()


## Output registry

In [ ]:
# Phase17C output registry and run summary.
from datetime import datetime
import json

table_files = sorted(str(p.relative_to(PROJECT_ROOT)) for p in (RELEASE_TABLE_DIR / "03_muller_brown_10d").glob("*.csv"))
figure_files = sorted(str(p.relative_to(PROJECT_ROOT)) for p in DIAGNOSTIC_FIG_DIR.glob(f"{FIG_PREFIX}*.pdf"))
registry = {
    "experiment": "10D Muller--Brown",
    "table_directory": str((RELEASE_TABLE_DIR / "03_muller_brown_10d").relative_to(PROJECT_ROOT)),
    "figure_directory": str(DIAGNOSTIC_FIG_DIR.relative_to(PROJECT_ROOT)),
    "tables": table_files,
    "figures": figure_files,
    "created_utc_like": datetime.utcnow().isoformat(timespec="seconds") + "Z",
}
registry_path = RELEASE_LOG_DIR / "03_muller_brown_10d_output_registry.json"
registry_path.write_text(json.dumps(registry, indent=2) + "\n", encoding="utf-8")
pd.DataFrame([{"experiment": "10D Muller--Brown", "n_tables": len(table_files), "n_figures": len(figure_files)}]).to_csv(
    RELEASE_TABLE_DIR / "03_muller_brown_10d_summary.csv", index=False
)
print("Phase17C registry written:", registry_path)
print("tables:", len(table_files), "figures:", len(figure_files))


In [ ]:
SCRIPTS_DIR = RELEASE_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))
from generate_canonical_release_figures import generate_muller_release
generate_muller_release()